# 3.4 Dynamic Tools (adaptado a lab local)

**Qué se ve aquí:** `@wrap_model_call` filtra **qué tools ve el modelo** según el runtime context (`request.override(tools=...)`). Usuario `external` → solo `web_search`; `internal` → también `sql_query`. ⚠️ Ocultar ≠ bloquear: ver el hallazgo de seguridad al final.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

### 📚 Teoría base del módulo 3: el ciclo del agente y dónde se engancha el middleware
`create_agent` es un **bucle**: el modelo piensa → (si pide tools) se ejecutan → el modelo vuelve a pensar → … hasta que responde sin pedir tools.

```
invoke ──► [before_agent] ──► ┌─► [before_model] ─► wrap_model_call( MODELO ) ─► [after_model] ─┐
                              │                                                                  │
                              │   ¿pidió tools? ── no ──► respuesta final ──► [after_agent] ──► fin
                              │        │ sí
                              └── wrap_tool_call( TOOL ) ◄┘
```
| Hook | Corre… | Ejemplo en este módulo |
|---|---|---|
| `before_agent` | 1 vez al empezar el `invoke` | borrar `ToolMessage`s (3.2) |
| `before_model` | antes de **cada** llamada al modelo | `SummarizationMiddleware` (3.2) |
| `wrap_model_call` | **envolviendo** cada llamada al modelo | cambiar modelo (3.4), tools (3.4, 3.5), prompt (`@dynamic_prompt` está hecho sobre esto) |
| `after_model` | después de cada respuesta del modelo | `HumanInTheLoopMiddleware` revisa los `tool_calls` y pausa (3.3) |
| `wrap_tool_call` | envolviendo cada ejecución de tool | reintentos o errores (como el interceptor MCP de M2.4) |

**Analogía:** el agente es un empleado; el middleware son los **supervisores** que revisan su trabajo en momentos fijos (al entrar, antes de hablar con el cliente, antes de enviar un correo…). El empleado no cambia; cambian las reglas que lo rodean.

### 📚 Teoría: tools dinámicas (qué utilería ve el modelo)
**Idea:** el middleware decide qué tools **ve** el modelo en cada llamada. Si se las quita, normalmente no las usará, porque no sabe que existen. ⚠️ **Pero ocultar no es bloquear:** si el modelo nombra una tool registrada en `create_agent`, el agente la ejecuta igual (lo comprobé en este notebook). El bloqueo real va en `@wrap_tool_call` (ver el final).

- Se hace con `@wrap_model_call` + **`request.override(tools=[...])`**.
- `create_agent(tools=[...])` debe tener **todas** las tools posibles (se registran en el grafo); el middleware elige **cuáles se muestran** en cada llamada.
- El criterio puede venir del **runtime context** (rol del usuario, este notebook) o del **state** (autenticado o no, en 3.5).

**Por qué importa:** menos tools visibles = menos tokens, menos confusión del modelo y menos riesgo (un usuario externo no llega a la base de datos interna). Es un *guardrail* típico de producción.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

### 📘 Las dos tools
- `web_search` (Tavily, `max_results=3` en este lab): información pública.
- `sql_query` (Chinook): información **interna**. Ojo: ejecuta cualquier SQL (ver M2.B).

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: Tavily sin límite devuelve mucho texto (latencia en CPU) → max_results=3.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.tools import tool
# from typing import Dict, Any
# from tavily import TavilyClient
# from langchain_community.utilities import SQLDatabase
#
# tavily_client = TavilyClient()
#
# db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")
#
#
# @tool
# def web_search(query: str) -> Dict[str, Any]:
#
#     """Search the web for information"""
#
#     return tavily_client.search(query)
#
# @tool
# def sql_query(query: str) -> str:
#
#     """Obtain information from the database using SQL queries"""
#
#     try:
#         return db.run(query)
#     except Exception as e:
#         return f"Error: {e}"

In [2]:
# 🟢 ADAPTADO LOCAL
# Motivo: Tavily sin límite devuelve mucho texto (latencia en CPU) → max_results=3.
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query, max_results=3)   # menos tokens en CPU

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

/tmp/ipykernel_837020/2544151272.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


### 📘 El rol viene en el runtime context
`UserRole.user_role`: `external` (por defecto) o `internal`. Lo pone quien llama al agente, no el modelo.

In [3]:
# Sin cambios
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

### 📘 El filtro
- `internal` → `pass`: no cambia nada, ve las dos tools.
- Cualquier otro → `request.override(tools=[web_search])`: solo ve la búsqueda web.

In [4]:
# Sin cambios
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     tools=[web_search, sql_query],
#     middleware=[dynamic_tool_call],
#     context_schema=UserRole
# )

In [5]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

### 📘 Qué esperar con `external`
La pregunta es sobre la base de datos, pero el modelo **no ve** `sql_query`: o busca en la web, o dice que no puede saberlo. Compáralo con la celda EXTRA de `internal` (abajo), que sí puede consultar la BD (en Chinook hay **275** artistas).

In [6]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I do not have access to a specific database, so I cannot tell you how many artists are in it.

Could you please provide me with more context? For example:

* **What database are you referring to?** (e.g., a specific music database, a database you are working with, etc.)
* **Where is this database located or how can I access it?**

If you can give me more information, I might be able to help you find the answer!


In [7]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

How many artists are in the database?
================================== Ai Message ==================================

I do not have access to a specific database, so I cannot tell you how many artists are in it.

Could you please provide me with more context? For example:

* **What database are you referring to?** (e.g., a specific music database, a database you are working with, etc.)
* **Where is this database located or how can I access it?**

If you can give me more information, I might be able to help you find the answer!


### 🟢 EXTRA (no está en el curso): el mismo agente con rol `internal`
Con `internal` el middleware no filtra y el modelo ve también `sql_query`. Compara con la respuesta de `external`.

In [11]:
# 🟢 ADAPTADO LOCAL — EXTRA: probar el rol interno
response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database data base field name is artist ?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

The database contains 275 artists.


In [12]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

How many artists are in the database data base field name is artist ?
================================== Ai Message ==================================
Tool Calls:
  sql_query (4c94a23c-f874-4f94-9ec6-239a465f38fc)
 Call ID: 4c94a23c-f874-4f94-9ec6-239a465f38fc
  Args:
    query: SELECT COUNT(*) FROM artist
================================= Tool Message =================================
Name: sql_query

[(275,)]
================================== Ai Message ==================================

The database contains 275 artists.


# 🟢 ADAPTADO LOCAL — 2º intento: el modelo conoce las tablas reales

In [13]:
# 🟢 ADAPTADO LOCAL — 2º intento: el modelo conoce las tablas reales
# Motivo: gemma4 consultó "artists" (no existe; es "Artist") y preguntó al usuario en vez de corregir.
tablas = db.get_usable_table_names()

agent_v2 = create_agent(
    model=MODEL,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole,
    system_prompt=f"""You are a helpful assistant.
    If you have the sql_query tool, the SQLite tables are: {tablas}. Use these exact names.
    If a query fails, read the error, fix the query and retry. Never ask the user for table names.""",
)

for rol in ["external", "internal"]:
    r = agent_v2.invoke(
        {"messages": [HumanMessage(content="How many artists are in the database? table is Artist")]},
        context={"user_role": rol},
    )
    usadas = [tc["name"] for m in r["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    print(f"{rol:<9} tools usadas: {usadas}")
    print(f"          respuesta: {r['messages'][-1].content[:200]}\n")

external  tools usadas: ['sql_query']
          respuesta: The database contains 275 artists.

internal  tools usadas: ['sql_query']
          respuesta: The database contains 275 artists.



In [14]:
# 🟢 ADAPTADO LOCAL — EXTRA: probar el rol interno
response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database? table is artist")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

The database contains 275 artists.


In [15]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

How many artists are in the database? table is artist
================================== Ai Message ==================================
Tool Calls:
  sql_query (2c47927e-386e-4023-9bee-37a074370c02)
 Call ID: 2c47927e-386e-4023-9bee-37a074370c02
  Args:
    query: SELECT COUNT(*) FROM artist
================================= Tool Message =================================
Name: sql_query

[(275,)]
================================== Ai Message ==================================

The database contains 275 artists.


## 🚨 Hallazgo de seguridad: ocultar una tool NO es bloquearla
En el 2º intento, el rol **`external` también usó `sql_query`** y respondió *"275 artists"*. ¿Cómo, si el middleware se la ocultó?

1. `request.override(tools=[web_search])` solo cambia **lo que ve el modelo** en esa llamada.
2. Pero `sql_query` sigue **registrada** en `create_agent(tools=[...])`, y el nodo que ejecuta tools corre **cualquier** tool registrada que el modelo nombre.
3. El nombre se filtró por el **system prompt** del 2º intento (*"If you have the sql_query tool…"*): el modelo la pidió por nombre aunque no la tuviera en su lista → **se ejecutó**.

Lo reproduje con un modelo simulado que pide la tool oculta: el SQL se ejecuta igual.

**Defensa en dos capas:**
| Capa | Hook | Qué hace |
|---|---|---|
| Ocultar | `@wrap_model_call` + `override(tools=...)` | El modelo no ve la tool (menos tokens, menos tentación) |
| **Bloquear** | `@wrap_tool_call` | Antes de **ejecutar**, revisa el rol y devuelve *"Acceso denegado"* si no tiene permiso |

> Regla de seguridad: el prompt **sugiere**, el código **impone**. Nunca nombres en el prompt tools que un rol no debe usar.

In [16]:
# 🟢 ADAPTADO LOCAL — 3er intento: ocultar (wrap_model_call) + BLOQUEAR (wrap_tool_call)
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

PERMISOS = {"external": {"web_search"}, "internal": {"web_search", "sql_query"}}

@wrap_tool_call
def guardia(request, handler):
    """Se ejecuta antes de CADA tool: si el rol no tiene permiso, no la corre."""
    rol = request.runtime.context.user_role
    nombre = request.tool_call["name"]
    if nombre not in PERMISOS.get(rol, set()):
        return ToolMessage(content=f"Acceso denegado: el rol '{rol}' no puede usar {nombre}.",
                           tool_call_id=request.tool_call["id"], name=nombre, status="error")
    return handler(request)

agent_v3 = create_agent(
    model=MODEL,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call, guardia],
    context_schema=UserRole,
    # el prompt ya NO nombra la tool; solo da las tablas por si el rol tiene acceso
    system_prompt=f"""You are a helpful assistant. Only use the tools you have been given.
    Database tables (only if you have database access): {db.get_usable_table_names()}.""",
)

for rol in ["external", "internal"]:
    r = agent_v3.invoke(
        {"messages": [HumanMessage(content="How many artists are in the database?")]},
        context={"user_role": rol},
    )
    usadas = [tc["name"] for m in r["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    bloqueos = [m.content for m in r["messages"] if m.type == "tool" and "Acceso denegado" in str(m.content)]
    print(f"{rol:<9} tools pedidas: {usadas}")
    if bloqueos:
        print(f"          ⛔ {bloqueos[0]}")
    print(f"          respuesta: {r['messages'][-1].content[:200]}\n")

external  tools pedidas: ['database_query']
          ⛔ Acceso denegado: el rol 'external' no puede usar database_query.
          respuesta: I am sorry, but I do not have the necessary permissions to access the database to count the number of artists.

internal  tools pedidas: ['sql_query']
          respuesta: The database contains 275 artists.



In [18]:
# 🟢 EXTRA: prueba de fuego del guardia (forzamos al modelo a pedir sql_query siendo external)
r = agent_v3.invoke(
    {"messages": [HumanMessage(content="Use the sql_query tool to run: SELECT COUNT(*) FROM Artist")]},
    context={"user_role": "external"},
)
for m in r["messages"]:
    m.pretty_print()

================================ Human Message =================================

Use the sql_query tool to run: SELECT COUNT(*) FROM Artist
================================== Ai Message ==================================
Tool Calls:
  sql_query (55453e0c-38cb-4347-b7c9-a8ada001ed18)
 Call ID: 55453e0c-38cb-4347-b7c9-a8ada001ed18
  Args:
    query: SELECT COUNT(*) FROM Artist
================================= Tool Message =================================
Name: sql_query

Acceso denegado: el rol 'external' no puede usar sql_query.
================================== Ai Message ==================================

I am sorry, but I encountered an access denial when trying to run the SQL query. The role I am using does not have permission to execute `sql_query`.


In [20]:
# 🟢 EXTRA: prueba de fuego del guardia (forzamos al modelo a pedir sql_query siendo external)
r = agent_v3.invoke(
    {"messages": [HumanMessage(content="Use the sql_query tool to run: SELECT COUNT(*) FROM Artist")]},
    context={"user_role": "internal"},
)
for m in r["messages"]:
    m.pretty_print()

================================ Human Message =================================

Use the sql_query tool to run: SELECT COUNT(*) FROM Artist
================================== Ai Message ==================================
Tool Calls:
  sql_query (fb24329e-4713-4dfa-82a6-318e5b67c3b5)
 Call ID: fb24329e-4713-4dfa-82a6-318e5b67c3b5
  Args:
    query: SELECT COUNT(*) FROM Artist
================================= Tool Message =================================
Name: sql_query

[(275,)]
================================== Ai Message ==================================

The total number of artists is 275.
